# 2장. Train Model (Finger Classification)
이 챕터에서는 수집한 손가락 데이터셋을 사용하여 CNN 모델(ResNet18)을 분류 학습시킬 것입니다.
손가락 개수(0 ~ 5)에 해당하는 6개의 클래스를 분류하도록 모델의 FC 레이어를 변경합니다.

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import PIL.Image
import os
import glob
import numpy as np

print('PyTorch and libraries imported successfully!')

## DataSet 및 DataLoader 정의
Jetbot 카메라에서 들어오는 BGR 이미지 형식과 일치하도록 RGB 채널을 BGR로 반전시키는 커스텀 Dataset 클래스를 정의합니다.

In [ ]:
DATASET_DIR = 'dataset_finger'
CLASSES = ['0', '1', '2', '3', '4', '5']

class FingerDataset(torch.utils.data.Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.image_paths = []
        self.labels = []
        for label in CLASSES:
            label_dir = os.path.join(directory, label)
            if os.path.isdir(label_dir):
                for img_name in os.listdir(label_dir):
                    if img_name.endswith('.jpg'):
                        self.image_paths.append(os.path.join(label_dir, img_name))
                        self.labels.append(int(label))
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = PIL.Image.open(image_path)
        label = self.labels[idx]
        
        # Augmentation & Resize
        image = self.color_jitter(image)
        image = transforms.functional.resize(image, (224, 224))
        image = transforms.functional.to_tensor(image)
        
        # Convert RGB to BGR to match live demo camera BGR format
        image = image.numpy()[::-1].copy()
        image = torch.from_numpy(image)
        
        image = transforms.functional.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return image, label

dataset = FingerDataset(DATASET_DIR)
print(f'Total dataset size: {len(dataset)}')

## 데이터셋 분할 및 DataLoader 생성
학습 데이터 90%, 검증 데이터 10% 비율로 무작위 분할합니다.

In [ ]:
test_percent = 0.1
num_test = int(test_percent * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [len(dataset) - num_test, num_test])

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)
print('DataLoader setup complete!')

## 모델 선언 및 FC 레이어 변경 (출력 클래스 6개)

In [ ]:
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, 6) # 0, 1, 2, 3, 4, 5 손가락 개수 (6개 클래스)
device = torch.device('cuda')
model = model.to(device)
print('Model loaded and configured for GPU!')

## 모델 훈련 및 저장
분류 문제를 위해 Cross-Entropy 손실 함수를 사용합니다. 에포크마다 검증 데이터의 손실이 낮아지면 최상의 모델을 `best_model_finger.pth`로 저장합니다.

In [ ]:
NUM_EPOCHS = 50
BEST_MODEL_PATH = 'best_model_finger.pth'
best_loss = 1e9

optimizer = optim.Adam(model.parameters())

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    for images, labels in iter(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        train_loss += float(loss)
        loss.backward()
        optimizer.step()
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)
    train_loss /= len(train_loader)
    train_acc = float(train_correct) / len(train_dataset)
    
    model.eval()
    test_loss = 0.0
    test_correct = 0
    with torch.no_grad():
        for images, labels in iter(test_loader):
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            test_loss += float(loss)
            _, preds = torch.max(outputs, 1)
            test_correct += torch.sum(preds == labels.data)
    test_loss /= len(test_loader)
    test_acc = float(test_correct) / len(test_dataset)
    
    print('Epoch %d: Train Loss: %.4f, Train Acc: %.4f, Test Loss: %.4f, Test Acc: %.4f' % 
          (epoch, train_loss, train_acc, test_loss, test_acc))
    
    if test_loss < best_loss:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_loss = test_loss

print('Success! Best model saved to best_model_finger.pth')